# C10-competition-craft — Practice p08 — Solution

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 20260804
ks = np.array([3, 5, 7, 9, 11, 15])

df = pd.read_csv("../data/train.csv")
FEATURES = [c for c in df.columns if c != "outcome"]
X = df[FEATURES]
y = df["outcome"].to_numpy()
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=150, random_state=SEED, stratify=y
)

scores = []
for k in ks:
    candidate = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=int(k))),
    ]).fit(X_tr, y_tr)
    scores.append(f1_score(y_val, candidate.predict(X_val), average="macro"))

val_f1s = np.array(scores, dtype=float)
best_k = int(ks[np.argmax(val_f1s)])
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=best_k)),
]).fit(X, y)


def predict_labels(X_test):
    return pd.Series(pipe.predict(X_test), index=X_test.index)


probe = X.iloc[350:390]
probe_out = predict_labels(probe)
contract_ok = bool(
    isinstance(probe_out, pd.Series)
    and len(probe_out) == len(probe)
    and probe_out.index.equals(probe.index)
    and set(probe_out.unique()) <= set(np.unique(y))
)
(val_f1s, best_k, contract_ok)

The sweep is judged only by validation macro-F1. Because `ks` is ascending, `np.argmax`'s first-maximum behavior also implements the smallest-(k) tie rule.

### Answer check

In [ ]:
expected = np.array([0.7766463326768818, 0.7665823769694612,
                     0.7939560439560439, 0.7893564476296547,
                     0.8103481812876873, 0.7939560439560439])
assert val_f1s.shape == (6,)
assert np.allclose(val_f1s, expected, atol=1e-12, rtol=0)
assert best_k == 11 and type(best_k) is int
assert pipe.named_steps["knn"].n_neighbors == best_k
assert contract_ok is True
assert probe_out.index.equals(probe.index)